In [0]:
%run ./sentinelmesh_agent.py

In [0]:
## Test Claude as the Agent Model

test_alert = (
    "Network flow alert: ICMP dominant traffic, 16529 packets per second, "
    "packet count 100, average packet length 60 bytes, packet length standard deviation 0, "
    "inter arrival time 0.00006 seconds, TCP flag activity: no TCP flags observed."
)

print("Testing Claude Sonnet as agent model")
print(run_triage(test_alert, model="anthropic-claude-sonnet-4"))

Testing Claude Sonnet as agent model
---

## 🚨 SentinelMesh Triage Report — AWAITING ANALYST APPROVAL

| Field | Detail |
|---|---|
| **Predicted Attack Family** | `DDOS_FLOOD` |
| **MITRE Technique** | T1498 — Network Denial of Service |
| **Severity Score** | 🔴 **10 / 10 (CRITICAL)** |
| **Urgency Tier** | CRITICAL |
| **Recommended Action** | Immediate containment |

---

### 🔬 Observable Evidence Summary
| Indicator | Value | Significance |
|---|---|---|
| Protocol | ICMP only | Classic flood vector; no connection state |
| Packet Rate | 16,529 pps | Far exceeds normal thresholds |
| Packet Size Std Dev | 0 bytes | Machine-generated, zero variance — automated tooling |
| Inter-Arrival Time | 0.00006s | Sub-millisecond, machine-precision cadence |
| TCP Flags | None | Confirms pure ICMP flood, no handshake evasion |

---

### 🛡️ Recommended Mitigations (MITRE T1498)
1. **Rate-limit ICMP** at the perimeter firewall immediately
2. **Engage upstream traffic filtering** / null-routing a

In [0]:
print("run_triage loaded:", "run_triage" in globals())
print(run_triage)

run_triage loaded: True
<function run_triage at 0xff0390455d00>


sentinelmesh_agent.py loaded successfully.


In [0]:
## Set Up the Evaluation Judge

# An LLM judge scores each triage on whether it correctly identified the
# attack family, comparing the agent's prediction against the hidden true
# label. This mirrors the judge-based evaluation pattern used to assess
# agent output quality.
import mlflow
import pandas as pd
from mlflow.genai.judges import make_judge

# load the family mapping so predictions and true labels are compared at the
# same level (family to family, not the 34 granular labels)
label_map = spark.table("main.default.cic_label_family_map").toPandas()
label_to_family = dict(zip(label_map["label"], label_map["attack_family"]))

print(f"Loaded family mapping for {len(label_to_family)} labels.")

Loaded family mapping for 34 labels.


In [0]:
## Build the Evaluation Dataset

# Each row carries the alert text in inputs (passed to the agent) and the
# expected attack family in expectations (the hidden ground truth used only
# by the judge). Several families are included for coverage across attack types.
import pandas as pd

sample = spark.table("main.default.cic_alerts_sample").toPandas()
sample["expected_family"] = sample["Label"].map(label_to_family)

families_to_eval = [
    "DDOS-ICMP_FLOOD", "DDOS-SYN_FLOOD", "DOS-UDP_FLOOD", "MIRAI-GREETH_FLOOD",
    "RECON-PORTSCAN", "VULNERABILITYSCAN", "MITM-ARPSPOOFING", "DNS_SPOOFING",
    "DICTIONARYBRUTEFORCE", "SQLINJECTION", "BACKDOOR_MALWARE", "BENIGN",
]

eval_rows = []
for fam in families_to_eval:
    matches = sample[sample["Label"] == fam]
    if len(matches) > 0:
        row = matches.iloc[0]
        eval_rows.append({
            "inputs": {"alert_text": row["alert_text"]},
            "expectations": {"expected_family": row["expected_family"]},
        })

eval_df = pd.DataFrame(eval_rows)
print(f"Evaluation dataset built with {len(eval_df)} alerts.")

Evaluation dataset built with 12 alerts.


## Automated Judge Limitation

The first automated judge attempt used a Databricks GPT OSS endpoint. That endpoint created MLflow evaluation traces, but earlier runs showed null or error assessments because the endpoint rejected the structured response fields used by MLflow judge calls.

The final automated judge section therefore uses the available Claude endpoint. The manual deterministic judge remains the primary reproducible evaluation because it does not depend on model endpoint support for structured judge output.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

for endpoint in w.serving_endpoints.list():
    name = endpoint.name
    if "claude" in name.lower() or "anthropic" in name.lower():
        print(name)

anthropic-claude-sonnet-4
databricks-claude-opus-4-8


In [0]:

# The evaluation first generates a fixed answer sheet from the SentinelMesh agent.
# Each row contains the original alert text, the expected attack family, and the agent response.
# This separates agent execution from judge evaluation, which makes the evaluation easier to inspect and reproduce.

## Generate Evaluation Answer Sheet

import pandas as pd
import time

AGENT_EVAL_MODEL = "anthropic-claude-sonnet-4"

answer_sheet_rows = []

def run_triage_with_retry(alert_text, model=AGENT_EVAL_MODEL, max_attempts=4, wait_seconds=20):
    for attempt in range(max_attempts):
        try:
            return str(run_triage(alert_text, model=model))
        except Exception as e:
            error_text = str(e)
            if "REQUEST_LIMIT_EXCEEDED" in error_text or "RateLimitError" in error_text or "429" in error_text:
                sleep_time = wait_seconds * (attempt + 1)
                print(f"Rate limit hit. Waiting {sleep_time} seconds before retry {attempt + 1}/{max_attempts}.")
                time.sleep(sleep_time)
            else:
                raise e

    return "ERROR: Agent response could not be generated after retry attempts due to rate limiting."

for i, row in enumerate(eval_rows):
    alert_text = row["inputs"]["alert_text"]
    expected_family = row["expectations"]["expected_family"]

    decision = run_triage_with_retry(alert_text)

    answer_sheet_rows.append({
        "inputs": {
            "alert_text": alert_text
        },
        "expectations": {
            "expected_family": expected_family
        },
        "outputs": decision,
        "agent_model": AGENT_EVAL_MODEL
    })

    print(f"[{i+1}/{len(eval_rows)}] generated response with {AGENT_EVAL_MODEL} for expected_family={expected_family}")
    time.sleep(10)

answer_sheet_df = pd.DataFrame(answer_sheet_rows)

display(answer_sheet_df)

[1/12] generated response with anthropic-claude-sonnet-4 for expected_family=DDOS_FLOOD
[2/12] generated response with anthropic-claude-sonnet-4 for expected_family=DDOS_FLOOD
[3/12] generated response with anthropic-claude-sonnet-4 for expected_family=DOS_FLOOD
[4/12] generated response with anthropic-claude-sonnet-4 for expected_family=MIRAI
[5/12] generated response with anthropic-claude-sonnet-4 for expected_family=RECON
[6/12] generated response with anthropic-claude-sonnet-4 for expected_family=VULNERABILITYSCAN
[7/12] generated response with anthropic-claude-sonnet-4 for expected_family=SPOOFING
[8/12] generated response with anthropic-claude-sonnet-4 for expected_family=SPOOFING
[9/12] generated response with anthropic-claude-sonnet-4 for expected_family=BRUTEFORCE
[10/12] generated response with anthropic-claude-sonnet-4 for expected_family=WEB_ATTACK
[11/12] generated response with anthropic-claude-sonnet-4 for expected_family=MALWARE
[12/12] generated response with anthropic

inputs,expectations,outputs,agent_model
"List(Network flow alert: ICMP dominant traffic (100% of flow), packets per second: 16,529, packet count: 100, average packet length: 60 bytes, packet length standard deviation: 0, minimum packet length: 60, maximum packet length: 60, total packet size: 60, total packet size sum: 6000, inter arrival time: 0.000061 seconds, SYN count: 0, ACK count: 0, RST count: 0, FIN count: 0, header length: 0, time to live: 64, variance: 0.00, highly uniform packet sizes, TCP flag activity: no TCP flags observed. The true label is not shown to the agent during evaluation.)",List(DDOS_FLOOD),"--- ## 🛡️ SentinelMesh Triage Report — AWAITING ANALYST APPROVAL | Field | Detail | |---|---| | **Predicted Attack Family** | `DOS_FLOOD` | | **MITRE Technique** | T1499 — Endpoint Denial of Service | | **Severity Score** | 🔴 **9 / 10 (CRITICAL)** | | **Urgency Tier** | CRITICAL | | **Recommended Action** | Immediate containment — apply connection limits, resource quotas, and application-layer filtering | --- ### 🔍 Analyst Reasoning | Observable Signal | Interpretation | |---|---| | **100% ICMP traffic** | No legitimate mixed-protocol session; pure flood vector | | **16,529 packets/sec** | Extremely high volumetric rate — well above normal ICMP thresholds | | **Zero packet size variance** (all 60B) | Machine-generated, scripted flood — not organic traffic | | **Inter-arrival time: 0.000061s** | ~16K pps confirmed; no human-paced behavior | | **No TCP flags (SYN/ACK/RST/FIN = 0)** | Confirms non-TCP, stateless flood; no handshake attempted | | **TTL: 64** | Consistent single-hop or spoofed origin; warrants source IP investigation | --- ### ⚠️ Recommended Immediate Actions (Pending Human Approval) 1. **Rate-limit or null-route** ICMP traffic from the source IP at the perimeter firewall/router. 2. **Apply ICMP flood thresholds** via connection-rate limiting on the affected endpoint. 3. **Investigate TTL=64** — determine if source IP is spoofed or reflects a nearby attacker node. 4. **Correlate with upstream flows** to check for distributed origin (potential DDoS escalation). 5. **Notify network operations** if the target host shows signs of resource exhaustion or availability degradation. > ✅ *All actions above require explicit human analyst approval before execution. No automated containment has been triggered.*",anthropic-claude-sonnet-4
"List(Network flow alert: TCP dominant traffic (100% of flow), packets per second: 44,502, packet count: 100, average packet length: 60 bytes, packet length standard deviation: 0, minimum packet length: 60, maximum packet length: 60, total packet size: 60, total packet size sum: 6000, inter arrival time: 0.000022 seconds, SYN count: 100, ACK count: 0, RST count: 0, FIN count: 0, header length: 20, time to live: 64, variance: 0.00, highly uniform packet sizes, TCP flag activity: SYN=1.00. The true label is not shown to the agent during evaluation.)",List(DDOS_FLOOD),"--- ## 🛡️ SentinelMesh Triage Report | Field | Detail | |---|---| | **Predicted Attack Family** | `DOS_FLOOD` — SYN Flood | | **MITRE Technique** | T1499 — Endpoint Denial of Service | | **Severity Score** | 🔴 **9 / 10 (CRITICAL)** | | **Urgency Tier** | CRITICAL | | **Recommended Action** | Immediate containment — connection limits, resource quotas, and application-layer filtering | --- ### 🔍 Analyst Reasoning | Observable | Significance | |---|---| | **TCP SYN = 1.00, ACK = 0** | Pure half-open connection flood — no handshake completion, classic SYN flood | | **44,502 packets/sec** | Extremely high rate, well above normal TCP session establishment thresholds | | **Packet size = 60 bytes (σ = 0)** | Zero variance, perfectly uniform — strongly indicative of synthetic/crafted packets | | **Inter-arrival time = 0.000022s** | ~22 µs between packets — machine-generated, not human-driven traffic | | **No RST / FIN / ACK** | No legitimate session teardown or establishment — purely offensive half-open state exhaustion | --- ###

In [0]:
## Fallback LLM Judge Without Structured Response Schema

import re
import time
import pandas as pd
from databricks.sdk import WorkspaceClient

try:
    openai_client
except NameError:
    openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

FALLBACK_LLM_JUDGE_MODEL = "anthropic-claude-sonnet-4"

VALID_FAMILIES_TEXT = """
DDOS_FLOOD, DOS_FLOOD, MIRAI, RECON, VULNERABILITYSCAN,
SPOOFING, BRUTEFORCE, WEB_ATTACK, MALWARE, BENIGN
"""

def get_expected_family(expectations):
    if isinstance(expectations, dict):
        return expectations.get("expected_family", "UNKNOWN")
    return str(expectations)

def parse_llm_verdict(text):
    text_lower = str(text).lower()

    match = re.search(r"verdict\s*:\s*(pass|fail)", text_lower)
    if match:
        return match.group(1)

    if text_lower.strip().startswith("pass"):
        return "pass"

    if text_lower.strip().startswith("fail"):
        return "fail"

    return "unknown"

def fallback_llm_judge_once(alert_text, expected_family, agent_output):
    judge_prompt = f"""
You are evaluating a SOC alert triage agent.

Your task:
Decide whether the agent predicted the exact expected attack family.

Valid attack families:
{VALID_FAMILIES_TEXT}

Alert input:
{alert_text}

Expected family:
{expected_family}

Agent output:
{agent_output}

Judging rule:
Return pass only if the agent's predicted attack family exactly matches the expected family.
Return fail if the predicted family is missing, unclear, null, errored, or different.
Ignore severity, MITRE wording, formatting, and recommendation quality.

Return exactly this format:
verdict: pass or fail
rationale: one short sentence
"""

    response = openai_client.chat.completions.create(
        model=FALLBACK_LLM_JUDGE_MODEL,
        messages=[
            {"role": "system", "content": "You are a strict SOC evaluation judge."},
            {"role": "user", "content": judge_prompt},
        ],
        temperature=0,
        max_tokens=200,
    )

    return str(response.choices[0].message.content)

def fallback_llm_judge_with_retry(alert_text, expected_family, agent_output, max_attempts=4, wait_seconds=15):
    for attempt in range(max_attempts):
        try:
            return fallback_llm_judge_once(alert_text, expected_family, agent_output)
        except Exception as e:
            error_text = str(e)
            if "REQUEST_LIMIT_EXCEEDED" in error_text or "RateLimitError" in error_text or "429" in error_text:
                sleep_time = wait_seconds * (attempt + 1)
                print(f"Rate limit hit. Waiting {sleep_time} seconds before retry {attempt + 1}/{max_attempts}.")
                time.sleep(sleep_time)
            else:
                raise e

    return "verdict: fail\nrationale: LLM judge could not complete due to endpoint rate limits."

llm_judge_rows = []

for i, row in answer_sheet_df.iterrows():
    alert_text = row["inputs"]["alert_text"]
    expected_family = get_expected_family(row["expectations"])
    agent_output = row["outputs"]

    judge_output = fallback_llm_judge_with_retry(
        alert_text=alert_text,
        expected_family=expected_family,
        agent_output=agent_output,
    )

    verdict = parse_llm_verdict(judge_output)

    llm_judge_rows.append({
        "expected_family": expected_family,
        "llm_judge_verdict": verdict,
        "llm_judge_output": judge_output,
        "agent_output_excerpt": str(agent_output)[:220],
    })

    print(f"[{i+1}/{len(answer_sheet_df)}] LLM judge verdict={verdict} expected_family={expected_family}")
    time.sleep(8)

llm_judge_df = pd.DataFrame(llm_judge_rows)

display(llm_judge_df)

pass_count = int((llm_judge_df["llm_judge_verdict"] == "pass").sum())
total_count = len(llm_judge_df)

print(f"Fallback LLM judge pass rate: {pass_count}/{total_count} = {pass_count / total_count:.0%}")

[1/12] LLM judge verdict=fail expected_family=DDOS_FLOOD
[2/12] LLM judge verdict=fail expected_family=DDOS_FLOOD
[3/12] LLM judge verdict=pass expected_family=DOS_FLOOD
[4/12] LLM judge verdict=fail expected_family=MIRAI
[5/12] LLM judge verdict=fail expected_family=RECON
[6/12] LLM judge verdict=fail expected_family=VULNERABILITYSCAN
[7/12] LLM judge verdict=fail expected_family=SPOOFING
[8/12] LLM judge verdict=fail expected_family=SPOOFING
[9/12] LLM judge verdict=fail expected_family=BRUTEFORCE
[10/12] LLM judge verdict=pass expected_family=WEB_ATTACK
[11/12] LLM judge verdict=pass expected_family=MALWARE
[12/12] LLM judge verdict=fail expected_family=BENIGN


expected_family,llm_judge_verdict,llm_judge_output,agent_output_excerpt
DDOS_FLOOD,fail,verdict: fail rationale: The agent predicted `DOS_FLOOD` but the expected attack family is `DDOS_FLOOD`.,--- ## 🛡️ SentinelMesh Triage Report — AWAITING ANALYST APPROVAL | Field | Detail | |---|---| | **Predicted Attack Family** | `DOS_FLOOD` | | **MITRE Technique** | T1499 — Endpoint Denial of Service | | **Severity Scor
DDOS_FLOOD,fail,verdict: fail rationale: The agent predicted DOS_FLOOD but the expected attack family is DDOS_FLOOD.,--- ## 🛡️ SentinelMesh Triage Report | Field | Detail | |---|---| | **Predicted Attack Family** | `DOS_FLOOD` — SYN Flood | | **MITRE Technique** | T1499 — Endpoint Denial of Service | | **Severity Score** | 🔴 **9 / 10
DOS_FLOOD,pass,"verdict: pass rationale: The agent explicitly predicted `DOS_FLOOD`, which exactly matches the expected attack family.",--- ## 🚨 SentinelMesh Triage Report — CRITICAL | Field | Detail | |---|---| | **Predicted Attack Family** | `DOS_FLOOD` | | **MITRE Technique** | T1499 — Endpoint Denial of Service | | **Severity Score** | 🔴 **9 / 10**
MIRAI,fail,"verdict: fail rationale: The agent predicted DOS_FLOOD, but the expected attack family is MIRAI.",--- ## 🛡️ SentinelMesh Triage Report — AWAITING ANALYST APPROVAL | Field | Detail | |---|---| | **Predicted Attack Family** | `DOS_FLOOD` | | **MITRE Technique** | T1499 — *Endpoint Denial of Service* | | **Severity Sc
RECON,fail,verdict: fail rationale: The agent predicted `WEB_ATTACK` but the expected attack family is `RECON`.,--- ## 🛡️ SentinelMesh Triage Report | Field | Detail | |---|---| | **Predicted Attack Family** | `WEB_ATTACK` | | **MITRE Technique** | T1190 — *Exploitation of Public-Facing Application* | | **Severity Score** | 🟠 **
VULNERABILITYSCAN,fail,"verdict: fail rationale: The agent predicted RECON, but the expected attack family is VULNERABILITYSCAN.",--- ## 🛡️ SentinelMesh Triage Report — Awaiting Analyst Approval | Field | Detail | |---|---| | **Predicted Attack Family** | `RECON` (UDP-based Active Scanning) | | **MITRE Technique** | T1595 — Active Scanning | | **
SPOOFING,fail,verdict: fail rationale: The agent predicted `DDOS_FLOOD` but the expected attack family is `SPOOFING`.,--- ## 🚨 SentinelMesh Triage Report — AWAITING ANALYST APPROVAL | Field | Detail | |---|---| | **Predicted Attack Family** | `DDOS_FLOOD` — ACK Flood | | **MITRE Technique** | T1498 — Network Denial of Service | | **Se
SPOOFING,fail,verdict: fail rationale: The agent predicted `DDOS_FLOOD` but the expected attack family is `SPOOFING`.,--- ## 🚨 SentinelMesh Triage Report — CRITICAL | Field | Detail | |---|---| | **Predicted Attack Family** | `DDOS_FLOOD` | | **MITRE Technique** | T1498 — Network Denial of Service | | **Severity Score** | 🔴 **10 / 10*
BRUTEFORCE,fail,"verdict: fail rationale: The agent predicted RECON, but the expected attack family is BRUTEFORCE.",--- ## 🛡️ SentinelMesh Triage Report — Awaiting Human Analyst Approval | Field | Detail | |---|---| | **Predicted Attack Family** | `RECON` (Active Scanning) | | **MITRE Technique** | T1595 — Active Scanning | | **Seve
WEB_ATTACK,pass,"verdict: pass rationale: The agent predicted `WEB_ATTACK`, which exactly matches the expected attack family.",--- ## 🛡️ SentinelMesh Triage Report — *Awaiting Human Analyst Approval* | Field | Detail | |---|---| | **Predicted Attack Family** | `WEB_ATTACK` | | **MITRE Technique** | T1190 — *Exploitation of Public-Facing Applic


Fallback LLM judge pass rate: 3/12 = 25%


In [0]:
## Documented MLflow make_judge Attempts

# This cell is intentionally documentation only.
# It is not executed in the final run because the available workspace endpoints
# could not complete MLflow make_judge scoring reliably.
#
# Attempted automated judge models:
#
# 1. databricks:/databricks-gpt-oss-120b
#    Result:
#    Failed because the endpoint hit workspace QPS limits and produced null or failed assessments.
#
# 2. databricks:/databricks-claude-opus-4-8
#    Result:
#    Failed because the endpoint was unavailable with a Databricks configured rate limit of zero.
#
# 3. databricks:/anthropic-claude-sonnet-4
#    Result:
#    Failed as an MLflow make_judge scorer because the Anthropic endpoint rejected
#    the structured response_schema field used by MLflow judge calls.
#
# These attempts confirm that the judge prompt was implemented, but the available
# serving endpoints were not compatible with structured make_judge scoring in this workspace.
#
# The notebook therefore keeps a fallback LLM judge using a normal Sonnet chat completion,
# plus a manual deterministic judge as the primary reproducible evaluation.

print("MLflow make_judge attempts documented. Final scoring uses fallback LLM judge plus manual deterministic judge.")

MLflow make_judge attempts documented. Final scoring uses fallback LLM judge plus manual deterministic judge.


In [0]:
## Manual Deterministic Judge

import re
import pandas as pd

VALID_FAMILIES = [
    "DDOS_FLOOD", "DOS_FLOOD", "VULNERABILITYSCAN", "BRUTEFORCE",
    "WEB_ATTACK", "SPOOFING", "MALWARE", "MIRAI", "RECON", "BENIGN",
]

COARSE = {
    "DDOS_FLOOD": "FLOOD",
    "DOS_FLOOD": "FLOOD",
    "MIRAI": "FLOOD",
    "RECON": "RECON_SCAN",
    "VULNERABILITYSCAN": "RECON_SCAN",
    "SPOOFING": "SPOOFING",
    "BRUTEFORCE": "INTRUSION",
    "WEB_ATTACK": "INTRUSION",
    "MALWARE": "INTRUSION",
    "BENIGN": "BENIGN",
}

def extract_predicted_family(text):
    text = str(text).upper()
    for family in VALID_FAMILIES:
        pattern = rf"(?<![A-Z_]){family}(?![A-Z_])"
        if re.search(pattern, text):
            return family
    return "UNKNOWN"

manual_results = []

for _, row in answer_sheet_df.iterrows():
    expected = row["expectations"]["expected_family"]
    output = row["outputs"]

    predicted = extract_predicted_family(output)

    exact_correct = predicted == expected
    coarse_correct = (
        predicted != "UNKNOWN" and COARSE.get(predicted) == COARSE.get(expected)
    )

    manual_results.append({
        "expected_family": expected,
        "predicted_family": predicted,
        "expected_coarse": COARSE.get(expected),
        "predicted_coarse": COARSE.get(predicted, "UNKNOWN"),
        "exact_correct": exact_correct,
        "coarse_correct": coarse_correct,
        "agent_output_excerpt": str(output)[:220],
    })

manual_eval_df = pd.DataFrame(manual_results)

display(manual_eval_df)

n = len(manual_eval_df)
exact_count = int(manual_eval_df["exact_correct"].sum())
coarse_count = int(manual_eval_df["coarse_correct"].sum())

print(f"Manual exact family accuracy: {exact_count}/{n} = {exact_count / n:.0%}")
print(f"Manual coarse triage accuracy: {coarse_count}/{n} = {coarse_count / n:.0%}")

expected_family,predicted_family,expected_coarse,predicted_coarse,exact_correct,coarse_correct,agent_output_excerpt
DDOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,true,true,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume ICMP flood) - **Severity Score:** 10 / 10 (critical) - **Recommended Action
DDOS_FLOOD,DOS_FLOOD,FLOOD,FLOOD,false,true,**Triage Summary** - **Predicted Attack Family:** **DOS_FLOOD** (SYN‑flood style denial‑of‑service) - **MITRE ATT&CK Technique:** **T1499 – Endpoint Denial of Service** – attacker overwhelms a single host/service with h
DOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,false,true,"**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume, uniform packet traffic to exhaust bandwidth/resources) - **Severity Score:**"
MIRAI,DDOS_FLOOD,FLOOD,FLOOD,false,true,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service – “Attacker overwhelms a target with high‑volume traffic to exhaust bandwidth or resources.”
RECON,BENIGN,RECON_SCAN,BENIGN,false,false,"**Triage Summary** - **Predicted Attack Family:** **BENIGN** - **MITRE ATT&CK Context:** Benign Traffic (no specific technique; normal, expected behavior) – *Mitigation: No action required; continue monitoring.* - *"
VULNERABILITYSCAN,RECON,RECON_SCAN,RECON_SCAN,false,true,"**Triage Summary** - **Predicted Attack Family:** RECON - **MITRE ATT&CK Technique:** T1595 – Active Scanning (probing network to discover hosts, open ports, and services) - **Severity Score:** 4 / 10 (Low) - **Re"
SPOOFING,DDOS_FLOOD,SPOOFING,FLOOD,false,false,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – *Network Denial of Service* – “Attacker overwhelms a target with high‑volume traffic to exhaust bandwidth or resources
SPOOFING,DDOS_FLOOD,SPOOFING,FLOOD,false,false,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume traffic overload) - **Severity Score:** 10 / 10 (Critical) - **Recommended
BRUTEFORCE,BENIGN,INTRUSION,BENIGN,false,false,**Triage Summary** - **Predicted Attack Family:** BENIGN - **MITRE ATT&CK Context:** Benign Traffic (no technique; normal expected behavior) – No mitigation needed beyond routine monitoring. - **Severity Score:** 1
WEB_ATTACK,MALWARE,INTRUSION,INTRUSION,false,true,**Triage Summary** - **Predicted Attack Family:** MALWARE - **MITRE ATT&CK Technique:** T1059 – Command and Scripting Interpreter (indicative of back‑door or C2 communication) - **Severity Score:** 9 / 10 (high seve


Manual exact family accuracy: 2/12 = 17%
Manual coarse triage accuracy: 7/12 = 58%


In [0]:
## Human Review Summary Table

human_review_df = manual_eval_df.copy()

human_review_df["human_feedback_value"] = human_review_df["exact_correct"].apply(
    lambda value: "pass" if value else "fail"
)

def build_human_rationale(row):
    if row["exact_correct"]:
        return (
            "The predicted family exactly matched the expected family. "
            "The trace is marked pass for exact family accuracy."
        )
    if row["coarse_correct"]:
        return (
            "The predicted family did not exactly match the expected family, "
            "so this is marked fail for exact family accuracy. The broader "
            "triage group was still correct."
        )
    return (
        "The predicted family and broader triage group did not match the expected result. "
        "The trace is marked fail."
    )

human_review_df["human_rationale"] = human_review_df.apply(build_human_rationale, axis=1)

display(human_review_df[[
    "expected_family",
    "predicted_family",
    "expected_coarse",
    "predicted_coarse",
    "human_feedback_value",
    "human_rationale"
]])

expected_family,predicted_family,expected_coarse,predicted_coarse,human_feedback_value,human_rationale
DDOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,pass,The predicted family exactly matched the expected family. The trace is marked pass for exact family accuracy.
DDOS_FLOOD,DOS_FLOOD,FLOOD,FLOOD,fail,"The predicted family did not exactly match the expected family, so this is marked fail for exact family accuracy. The broader triage group was still correct."
DOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,fail,"The predicted family did not exactly match the expected family, so this is marked fail for exact family accuracy. The broader triage group was still correct."
MIRAI,DDOS_FLOOD,FLOOD,FLOOD,fail,"The predicted family did not exactly match the expected family, so this is marked fail for exact family accuracy. The broader triage group was still correct."
RECON,BENIGN,RECON_SCAN,BENIGN,fail,The predicted family and broader triage group did not match the expected result. The trace is marked fail.
VULNERABILITYSCAN,RECON,RECON_SCAN,RECON_SCAN,fail,"The predicted family did not exactly match the expected family, so this is marked fail for exact family accuracy. The broader triage group was still correct."
SPOOFING,DDOS_FLOOD,SPOOFING,FLOOD,fail,The predicted family and broader triage group did not match the expected result. The trace is marked fail.
SPOOFING,DDOS_FLOOD,SPOOFING,FLOOD,fail,The predicted family and broader triage group did not match the expected result. The trace is marked fail.
BRUTEFORCE,BENIGN,INTRUSION,BENIGN,fail,The predicted family and broader triage group did not match the expected result. The trace is marked fail.
WEB_ATTACK,MALWARE,INTRUSION,INTRUSION,fail,"The predicted family did not exactly match the expected family, so this is marked fail for exact family accuracy. The broader triage group was still correct."


In [0]:
## Select Representative Traces for Manual Review

print("Exact pass examples:")
display(manual_eval_df[manual_eval_df["exact_correct"] == True])

print("Fail but coarse correct examples:")
display(manual_eval_df[
    (manual_eval_df["exact_correct"] == False) &
    (manual_eval_df["coarse_correct"] == True)
].head(3))

print("Fail and coarse incorrect examples:")
display(manual_eval_df[
    (manual_eval_df["exact_correct"] == False) &
    (manual_eval_df["coarse_correct"] == False)
].head(3))

Exact pass examples:


expected_family,predicted_family,expected_coarse,predicted_coarse,exact_correct,coarse_correct,agent_output_excerpt
DDOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,true,true,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume ICMP flood) - **Severity Score:** 10 / 10 (critical) - **Recommended Action
MALWARE,MALWARE,INTRUSION,INTRUSION,true,true,**Triage Summary** - **Predicted Attack Family:** MALWARE - **MITRE ATT&CK Technique:** T1059 – Command and Scripting Interpreter (indicative of low‑and‑slow C2 traffic) - **Severity Score:** 9 / 10 (high) - **Rec


Fail but coarse correct examples:


expected_family,predicted_family,expected_coarse,predicted_coarse,exact_correct,coarse_correct,agent_output_excerpt
DDOS_FLOOD,DOS_FLOOD,FLOOD,FLOOD,false,true,**Triage Summary** - **Predicted Attack Family:** **DOS_FLOOD** (SYN‑flood style denial‑of‑service) - **MITRE ATT&CK Technique:** **T1499 – Endpoint Denial of Service** – attacker overwhelms a single host/service with h
DOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,false,true,"**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume, uniform packet traffic to exhaust bandwidth/resources) - **Severity Score:**"
MIRAI,DDOS_FLOOD,FLOOD,FLOOD,false,true,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service – “Attacker overwhelms a target with high‑volume traffic to exhaust bandwidth or resources.”


Fail and coarse incorrect examples:


expected_family,predicted_family,expected_coarse,predicted_coarse,exact_correct,coarse_correct,agent_output_excerpt
RECON,BENIGN,RECON_SCAN,BENIGN,false,false,"**Triage Summary** - **Predicted Attack Family:** **BENIGN** - **MITRE ATT&CK Context:** Benign Traffic (no specific technique; normal, expected behavior) – *Mitigation: No action required; continue monitoring.* - *"
SPOOFING,DDOS_FLOOD,SPOOFING,FLOOD,false,false,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – *Network Denial of Service* – “Attacker overwhelms a target with high‑volume traffic to exhaust bandwidth or resources
SPOOFING,DDOS_FLOOD,SPOOFING,FLOOD,false,false,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume traffic overload) - **Severity Score:** 10 / 10 (Critical) - **Recommended


In [0]:
## Log Final Evaluation Metrics

import mlflow

exact_accuracy = exact_count / n
coarse_accuracy = coarse_count / n

fallback_llm_pass_rate = None
if "llm_judge_df" in globals():
    fallback_llm_pass_count = int((llm_judge_df["llm_judge_verdict"] == "pass").sum())
    fallback_llm_total = len(llm_judge_df)
    fallback_llm_pass_rate = fallback_llm_pass_count / fallback_llm_total

with mlflow.start_run(run_name="sentinelmesh_final_evaluation_summary"):
    mlflow.log_params({
        "agent_model_final_eval": "anthropic-claude-sonnet-4",
        "primary_evaluation_method": "manual_deterministic_judge",
        "secondary_evaluation_method": "fallback_llm_judge",
        "make_judge_status": "attempted_but_endpoint_limited",
        "eval_dataset_size": n,
        "exact_family_metric": "predicted_family == expected_family",
        "coarse_metric": "operational triage group match",
    })

    metrics = {
        "manual_exact_family_accuracy": exact_accuracy,
        "manual_coarse_triage_accuracy": coarse_accuracy,
        "manual_exact_correct_count": exact_count,
        "manual_coarse_correct_count": coarse_count,
        "eval_total_count": n,
    }

    if fallback_llm_pass_rate is not None:
        metrics["fallback_llm_judge_pass_rate"] = fallback_llm_pass_rate
        metrics["fallback_llm_judge_pass_count"] = fallback_llm_pass_count
        metrics["fallback_llm_judge_total_count"] = fallback_llm_total

    mlflow.log_metrics(metrics)

print("Final evaluation metrics logged to MLflow.")
print(f"Manual exact family accuracy: {exact_count}/{n} = {exact_accuracy:.0%}")
print(f"Manual coarse triage accuracy: {coarse_count}/{n} = {coarse_accuracy:.0%}")

if fallback_llm_pass_rate is not None:
    print(f"Fallback LLM judge pass rate: {fallback_llm_pass_count}/{fallback_llm_total} = {fallback_llm_pass_rate:.0%}")

Final evaluation metrics logged to MLflow.
Manual exact family accuracy: 2/12 = 17%
Manual coarse triage accuracy: 7/12 = 58%
Fallback LLM judge pass rate: 3/12 = 25%


# **Final Evaluation Summary**

SentinelMesh was evaluated with automated, deterministic, and human review methods.

The final answer sheet was generated with anthropic-claude-sonnet-4 as the agent model. Earlier agent runs with databricks-gpt-oss-120b encountered workspace QPS rate limits, which produced null responses and failed traces. Sonnet completed the triage workflow successfully and was therefore used for the final reproducible answer sheet.

An automated LLM judge was implemented in two ways. First, MLflow make_judge was attempted with Databricks GPT OSS and Claude based endpoints. These attempts showed that the judge prompt structure was valid, but the available endpoints produced rate limit errors, disabled endpoint errors, or structured response_schema compatibility failures. Second, a fallback LLM judge was implemented using a normal Sonnet chat completion. This provided an automated second opinion without requiring MLflow structured judge output support.

The final reproducible score uses the manual deterministic judge. This judge extracts the predicted family from the agent response using exact family matching and compares it directly against the expected family.

The final exact family accuracy was 2 out of 12, or 17 percent. This strict score measures whether SentinelMesh predicted the exact mapped attack family.

The final coarse triage accuracy was 7 out of 12, or 58 percent. This broader score measures whether SentinelMesh placed the alert into the correct operational group, such as flood, reconnaissance or scan, spoofing, intrusion, or benign.

These results show that SentinelMesh is stronger as a Tier 1 SOC triage assistant than as a precise standalone classifier. The agent can often recognize the broader operational pattern, but exact family classification remains difficult because several IoT flow based attacks share similar observable traffic characteristics.

Human review remains part of the evaluation process. SentinelMesh does not execute containment actions directly. A SOC analyst reviews the trace, predicted family, severity score, MITRE context, and recommended action before approval. Human review is also used to inspect disagreement cases between the fallback LLM judge and the deterministic judge.